# R2 · 1 · Avaliação complementar: pico e entressafra (A), calibração por horizonte e fase (D2), pesos de ensemble (E4)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas anteriores, inclusive `semanal_lib.py`):
1. suba **`r2_lib.py`** (pasta `data_imdc_2026/RODADA2/`);
2. **crie à mão** as pastas `resultados_semanal_r2`, `resultados_uf_r2`, `cache_r2` e `avaliacao_r2` (evita pastas duplicadas quando há duas sessões abertas).

**Nada antigo é alterado.** Os modelos novos gravam em `resultados_semanal_r2/` e `resultados_uf_r2/`; as tabelas novas vão para `avaliacao_r2/` com a data e a hora no nome (`..._AAAAMMDD-HHMM.csv`) e uma cópia `..._ultimo.csv`. As tabelas da rodada anterior (`resultados_semanal_avaliacao/`) continuam como estão.

**Retomada:** se a internet ou a energia cair, reconecte e **rode tudo de novo**. Cada modelo × rodada/temporada vira um arquivo assim que termina (gravado como `.tmp` e renomeado); os treinos longos das redes gravam um checkpoint a cada 5 épocas e continuam da última época salva; o tsfresh grava uma UF por vez; o Optuna grava o estudo no Drive a cada tentativa; o Chronos ajustado fica salvo e não é refeito.

CPU, 15–40 min. **Rode agora** (avalia os modelos que já existem) e **de novo no fim**, depois dos notebooks 2–5, para incluir os modelos novos. Cada execução grava um novo conjunto de tabelas com carimbo de data; as anteriores ficam.

**Fases** (por UF, limiares calculados só com dados até 2017): *entressafra* = semana abaixo do percentil 25 da incidência; *alta (pico)* = acima do percentil 75; *intermediária* = entre os dois. **Pico:** erro na semana do pico (semanas) e na altura do pico (%), por UF × temporada. **Cobertura por quantil:** % dos casos abaixo de cada quantil (o ideal é o nível do quantil).


In [ ]:
NOTEBOOK = 'r2_avaliacao'
PACOTES = ''

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r2', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'dados', 'dados_uf',
                 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_SEM_R2 = os.path.join(PASTA_DRIVE, 'resultados_semanal_r2')
PASTA_UF_R2 = os.path.join(PASTA_DRIVE, 'resultados_uf_r2')
PASTA_CACHE = os.path.join(PASTA_DRIVE, 'cache_r2')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
ARQ_SEL = os.path.join(PASTA_DRIVE, 'resultados_semanal', 'selecao_grupos.json')
for p in (PASTA_SEM_R2, PASTA_UF_R2, PASTA_CACHE, PASTA_AV2, os.path.join(PASTA_CACHE, 'ckpt')):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2
for m in (V, L, CM, U, S, R2):
    importlib.reload(m)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
except ImportError:
    torch = None
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
RODADAS = S.rodadas(D, felipe=True)
LIM = R2.limiares(D)                        # p25/p75 de cada UF com dados até 2017 (antes da afinação)
REFAZER = False


## A · Semanal


In [ ]:
# @title A1 · Ler todas as previsões semanais (rodada anterior + rodada 2) e os ensembles de referência
PASTAS_SEM = sorted(p for p in glob.glob(os.path.join(PASTA_DRIVE, 'resultados_semanal*'))
                    if os.path.isdir(p) and not p.endswith('_avaliacao'))
print('Pastas:', [os.path.basename(p) for p in PASTAS_SEM])
P0 = S.ler_semanal(PASTAS_SEM)
P0['date'], P0['origem'] = pd.to_datetime(P0['date']), pd.to_datetime(P0['origem'])
def ens_sem(membros, nome):
    global P0
    if all(m in set(P0.modelo) for m in membros):
        P0 = pd.concat([P0, S.ensemble_semanal(P0, membros, nome)], ignore_index=True)
    else:
        print(f'{nome}: faltam {[m for m in membros if m not in set(P0.modelo)]}')
ens_sem(['GBM_xgb_todas_sem', 'ARIMA_Fourier_sem'], 'ENS_GBM_ARIMA_sem')                         # o vencedor anterior
ens_sem(['GBM_xgb_todas_sem', 'ARIMA_Fourier_sem', 'MTGNN_cresc_sem'], 'ENS_GBM_ARIMA_MTGNNcresc_sem')  # corrige a falha
PV = S.com_verdade(P0, D)
PV['fase_origem'] = R2.fase_origem(PV, D, LIM)
PV['balde'] = PV.h.astype(str) + '|' + PV.fase_origem
print(len(PV), 'linhas ·', PV.modelo.nunique(), 'modelos')


In [ ]:
# @title A2 · D2: calibração por horizonte e fase na origem (λ = largura, δ = deslocamento da mediana), escolhida na afinação
ARQ_CAL = os.path.join(PASTA_AV2, 'calibracao_semanal.json')
CAL_SEM = {} if REFAZER else R2.json_le(ARQ_CAL, {})
for m in sorted(PV.modelo.unique()):
    if m in CAL_SEM or m in S.REFS_SEM:
        continue
    T = PV[(PV.modelo == m) & (PV.rodada.map(S.grupo_rodada) == 'afinacao')]
    if T.rodada.nunique() < len(TUNING):
        continue
    CAL_SEM[m] = R2.calibrar(T, 'balde')
    R2.json_grava(CAL_SEM, ARQ_CAL)                                   # grava a cada modelo (retoma)
if os.path.exists(ARQ_CAL):
    shutil.copy(ARQ_CAL, ARQ_CAL.replace('.json', f'_{R2.carimbo()}.json'))
PCAL = pd.concat([R2.aplicar_cal(PV[PV.modelo == m], 'balde', t) for m, t in CAL_SEM.items() if (PV.modelo == m).any()],
                 ignore_index=True)
ex = 'ENS_GBM_ARIMA_sem'
if ex in CAL_SEM:
    display(pd.DataFrame(CAL_SEM[ex]).T.round(3))


In [ ]:
# @title A3 · E4: pesos de ensemble por horizonte e fase (afinação) → ENS_stack_sem
CAND_SEM = ['GBM_xgb_todas_sem', 'ARIMA_Fourier_sem', 'MTGNN_cresc_sem', 'GBM_xgb_fase_sem', 'GBM_xgb_tsf_sem',
            'GBM_xgb_fase_L1_sem', 'MTGNN_cresc_escala_sem', 'GBM_xgb_optuna_sem', 'Chronos2FT_sem',
            'MTGNN_cresc_fase_sem', 'GBM_xgb_optuna_tsf_sem', 'ARIMAX_fase_sem']
MEMB_SEM = [m for m in CAND_SEM if m in set(PV.modelo)]
print('Membros:', MEMB_SEM)
CH = ['rodada', 'h', 'date', 'uf_code', 'balde']
T = PV[PV.rodada.map(S.grupo_rodada) == 'afinacao']
PESOS_SEM = R2.pesos_stacking(T, MEMB_SEM, CH, 'balde')
R2.json_grava(PESOS_SEM, os.path.join(PASTA_AV2, f'pesos_stack_semanal_{R2.carimbo()}.json'))
ST = R2.aplicar_stacking(PV, MEMB_SEM, CH, 'balde', PESOS_SEM, 'ENS_stack_sem')
info = PV.drop_duplicates(['rodada', 'h', 'date', 'uf_code'])[['rodada', 'h', 'date', 'uf_code', 'y', 'season', 'origem', 'fase_origem']]
ST = ST.merge(info, on=['rodada', 'h', 'date', 'uf_code'])
PA = R2.com_fase(pd.concat([PV, PCAL, ST], ignore_index=True), D, LIM)
pd.DataFrame(PESOS_SEM).T.round(2)


In [ ]:
# @title A4 · Tabelas semanais (gerais, por fase, pico, cobertura) com carimbo
TAB = {}
for g in ('afinacao', 'teste', 'felipe'):
    TAB[g] = S.resumo_semanal(PA, g)
    R2.salvar_tabela(TAB[g], PASTA_AV2, f'semanal_{g}')
TT = PA[PA.rodada.map(S.grupo_rodada) == 'teste']
FASE_SEM = R2.metricas_fase(TT[TT.h.isin([1, 4])], ['modelo', 'h'])
PICO_SEM = R2.erros_pico(TT[TT.h == 1], 'rodada')
COB_SEM = R2.cobertura_quantis(TT[TT.h == 1])
for nome, t in (('semanal_teste_por_fase', FASE_SEM), ('semanal_teste_pico_h1', PICO_SEM), ('semanal_teste_cobertura_h1', COB_SEM)):
    R2.salvar_tabela(t, PASTA_AV2, nome)
# visão resumida: régua geral + desempenho no pico e na entressafra (h = 1)
f1 = FASE_SEM[FASE_SEM.h == 1].pivot_table(index='modelo', columns='fase', values=['WIS_log', 'vies_pct', 'IC90'])
f1.columns = [f'{a}_{b}' for a, b in f1.columns]
RES_SEM = TAB['teste'].query('h == 1')[['modelo', 'R_vs_Climatologia_sem', 'MAE', 'WAPE', 'IC90']].merge(
    f1.reset_index(), on='modelo', how='left').merge(PICO_SEM, on='modelo', how='left').sort_values('R_vs_Climatologia_sem')
R2.salvar_tabela(RES_SEM, PASTA_AV2, 'semanal_resumo_fases_h1')
RES_SEM.head(25).round(3)


## B · Temporada


In [ ]:
# @title B1 · Ler todos os modelos de temporada (rodadas anteriores + resultados_uf_prox + resultados_uf_r2), λ dos novos
PASTAS_UF = [p for p in sorted(glob.glob(os.path.join(PASTA_DRIVE, 'resultados_uf*'))) if os.path.isdir(p)]
FONTES = [('região→UF', PASTA_COMP)] + [(os.path.basename(p), p) for p in PASTAS_UF] + [('externo', PASTA_EXT)]
PU = L.carregar_previsoes(FONTES, ctx.Du.ufs, 'uf', mostrar=False)
NOVOS_PROX = sorted(PU[PU.fonte.str.contains('prox')].modelo.unique())
NOVOS_R2 = sorted(PU[PU.fonte.str.contains('_r2')].modelo.unique())
NOVOS = NOVOS_PROX + NOVOS_R2
PU = PU[~PU.modelo.str.startswith('ENS')][L.COLS_PREV]
ens = [os.path.join(p, 'ensembles_uf.parquet') for p in PASTAS_UF if os.path.exists(os.path.join(p, 'ensembles_uf.parquet'))]
ENS0 = pd.read_parquet(ens[0])[L.COLS_PREV] if ens else PU.iloc[:0]
REF = ctx.ref[L.COLS_PREV] if ctx.ref is not None else PU.iloc[:0]
arq_f = os.path.join(PASTA_DRIVE, 'COMPARACAO_FELIPE', 'felipe_no_meu_jogo_previsoes.parquet')
FEL = pd.read_parquet(arq_f) if os.path.exists(arq_f) else PU.iloc[:0]
FEL = FEL[FEL.modelo.isin(['Felipe_XGB_ponto', 'Felipe_XGB_quantis'])][L.COLS_PREV] if len(FEL) else FEL
print('Novos (prox + r2):', NOVOS)
ARQ_LAM = os.path.join(PASTA_AV2, 'lambda_temporada.json')
LAM_T = {} if REFAZER else R2.json_le(ARQ_LAM, {})
clim = PU[(PU.modelo == 'climatologia') & PU.season.isin(TUNING)]
for m in NOVOS:
    if m in LAM_T:
        continue
    g = PU[(PU.modelo == m) & PU.season.isin(TUNING)]
    if g.season.nunique() < len(TUNING):
        print('sem afinação completa:', m); continue
    rs = []
    for l in (0.7, 0.8, 0.9, 1.0, 1.15, 1.3, 1.5):
        _, rr = V.avaliar(pd.concat([U.ajusta_largura(g, l).assign(modelo='x'), clim]), ctx.ver_uf, ref=('climatologia',))
        rs.append(float(rr.set_index('modelo').loc['x', 'R_vs_climatologia']))
    LAM_T[m] = (0.7, 0.8, 0.9, 1.0, 1.15, 1.3, 1.5)[int(np.argmin(rs))]
    R2.json_grava(LAM_T, ARQ_LAM)
CAL2 = [U.ajusta_largura(PU[PU.modelo == m], l).assign(modelo=f'{m}+λ') for m, l in LAM_T.items() if l != 1.0 and (PU.modelo == m).any()]
TODOS = pd.concat([REF, PU, ENS0, FEL] + CAL2, ignore_index=True).drop_duplicates(['modelo', 'season', 'uf_code', 'date'], keep='last')
print('λ:', LAM_T)


In [ ]:
# @title B2 · Ensembles de temporada: grafos com λ, + melhor MTGNN da rodada anterior, + melhor da rodada 2
def ens(membros, nome):
    global TODOS
    falta = [m for m in membros if m not in set(TODOS.modelo)]
    if falta:
        print(f'{nome}: faltam {falta}'); return
    e, _ = V.ensemble(TODOS, membros, nome, metodo='mediana')
    TODOS = pd.concat([TODOS, e], ignore_index=True)
BASE_G = ['MTGNN_UF+λ', 'A3TGCN_UF+λ', 'GCNGRU_UF+λ']
ens(BASE_G, 'ENS_grafos_λ')
def melhor_afinacao(cands):
    cands = [m for m in cands if m in set(TODOS.modelo)]
    if not cands:
        return None
    _, ra = V.avaliar(TODOS[TODOS.season.isin(TUNING) & TODOS.modelo.isin(cands + ['climatologia'])][L.COLS_PREV], ctx.ver_uf, ref=('climatologia',))
    ra = ra[(ra.temporadas == len(TUNING)) & ra.modelo.isin(cands)].dropna(subset=['R_vs_climatologia'])
    return ra.sort_values('R_vs_climatologia').modelo.iloc[0] if len(ra) else None
b_prox = melhor_afinacao([m + s for m in NOVOS_PROX for s in ('', '+λ')])
if b_prox:
    ens(BASE_G + [b_prox], 'ENS_grafos2_λ')
b_todos = melhor_afinacao([m + s for m in NOVOS for s in ('', '+λ')])
if b_todos:
    ens(BASE_G + [b_todos], 'ENS_grafos3_λ')
print('melhor novo (rodada anterior):', b_prox, '· melhor novo (todos):', b_todos)


In [ ]:
# @title B3 · D2 e E4 na temporada: calibração por bloco de antecedência e pesos por bloco (afinação)
c_data = {s: pd.Timestamp(D.datas[D.corte(s)]) for s in TUNING + TESTE}
TV = S.com_verdade(TODOS[TODOS.season.isin(TUNING + TESTE)], D)
TV['h'] = ((TV.date - TV.season.map(c_data)).dt.days // 7).astype(int)
TV['balde'] = R2.bloco_horizonte(TV.h.values)
_, ra = V.avaliar(TODOS[TODOS.season.isin(TUNING)][L.COLS_PREV], ctx.ver_uf, ref=('climatologia',))
TOP_AF = ra[ra.temporadas == len(TUNING)].dropna(subset=['R_vs_climatologia']).sort_values('R_vs_climatologia').modelo.head(30).tolist()
ARQ_CALT = os.path.join(PASTA_AV2, 'calibracao_temporada.json')
CAL_T = {} if REFAZER else R2.json_le(ARQ_CALT, {})
for m in TOP_AF:
    if m in CAL_T:
        continue
    CAL_T[m] = R2.calibrar(TV[(TV.modelo == m) & TV.season.isin(TUNING)], 'balde', min_linhas=50)
    R2.json_grava(CAL_T, ARQ_CALT)
TCAL = pd.concat([R2.aplicar_cal(TV[TV.modelo == m], 'balde', t) for m, t in CAL_T.items() if (TV.modelo == m).any()], ignore_index=True)
CAND_T = ['MTGNN_pesopop_UF+λ', 'MTGNN_UF+λ', 'A3TGCN_UF+λ', 'GCNGRU_UF+λ', 'TFT_pool_UF', 'DoisEstagios_pool_UF'] + ([b_todos] if b_todos else [])
MEMB_T = list(dict.fromkeys(m for m in CAND_T if m in set(TV.modelo)))
CHT = ['season', 'date', 'uf_code', 'balde']
PESOS_T = R2.pesos_stacking(TV[TV.season.isin(TUNING)], MEMB_T, CHT, 'balde')
R2.json_grava(PESOS_T, os.path.join(PASTA_AV2, f'pesos_stack_temporada_{R2.carimbo()}.json'))
STT = R2.aplicar_stacking(TV, MEMB_T, CHT, 'balde', PESOS_T, 'ENS_stack_UF')
STT = STT.merge(TV.drop_duplicates(['season', 'date', 'uf_code'])[['season', 'date', 'uf_code', 'y', 'h']], on=['season', 'date', 'uf_code'])
TV2 = pd.concat([TV, TCAL, STT], ignore_index=True)
print('membros do stacking:', MEMB_T)
pd.DataFrame(PESOS_T).T.round(2)


In [ ]:
# @title B4 · Tabelas da temporada (régua oficial, fases, pico, cobertura) com carimbo
TT2 = TV2[TV2.season.isin(TESTE)]
_, RES_T = V.avaliar(TT2[L.COLS_PREV], ctx.ver_uf, ref=('BB', 'climatologia'))
RES_T = RES_T[RES_T.temporadas == len(TESTE)]
pont = pd.DataFrame([dict(modelo=m, **S.metricas(g)) for m, g in TT2.groupby('modelo')])
TAB_T = RES_T[[c for c in ['modelo', 'R_vs_BB', 'UFs_melhor_que_BB', 'R_vs_climatologia'] if c in RES_T]].merge(pont, on='modelo')
TTF = R2.com_fase(TT2, D, LIM)
FASE_T = R2.metricas_fase(TTF, ['modelo'])
PICO_T = R2.erros_pico(TT2, 'season')
COB_T = R2.cobertura_quantis(TT2)
f1 = FASE_T.pivot_table(index='modelo', columns='fase', values=['WIS_log', 'vies_pct', 'IC90'])
f1.columns = [f'{a}_{b}' for a, b in f1.columns]
RES_TEMP = TAB_T.merge(f1.reset_index(), on='modelo', how='left').merge(PICO_T, on='modelo', how='left') \
    .sort_values('R_vs_BB' if 'R_vs_BB' in TAB_T else 'WIS')
for nome, t in (('temporada_teste', TAB_T), ('temporada_teste_por_fase', FASE_T), ('temporada_teste_pico', PICO_T),
                ('temporada_teste_cobertura', COB_T), ('temporada_resumo_fases', RES_TEMP)):
    R2.salvar_tabela(t, PASTA_AV2, nome)
RES_TEMP.head(30).round(3)


## C · Os dois níveis lado a lado e o top 5 de cada métrica (inclusive pico e entressafra)


In [ ]:
# @title C1 · Tabela única (teste) e top 5
cols = ['modelo', 'nivel', 'R_vs_clim', 'WIS', 'WIS_log', 'MAE', 'WAPE', 'sMAPE', 'IC90', 'vies_pct',
        'WIS_log_alta (pico)', 'WIS_log_entressafra', 'erro_semana_abs_medio', 'erro_altura_mediano_pct', 'pico_no_IC90_pct']
t1 = RES_TEMP.rename(columns={'R_vs_climatologia': 'R_vs_clim'}).assign(nivel='temporada (SE25)')
ts = RES_SEM.rename(columns={'R_vs_Climatologia_sem': 'R_vs_clim'}).assign(nivel='semanal h=1')
t4 = TAB['teste'].query('h == 4').rename(columns={'R_vs_Climatologia_sem': 'R_vs_clim'}).assign(nivel='semanal h=4')
UNICA = pd.concat([t1, ts, t4], ignore_index=True)
UNICA = UNICA[[c for c in cols if c in UNICA]]
R2.salvar_tabela(UNICA, PASTA_AV2, 'tabela_unica_teste')
crit = {'R_vs_clim': True, 'WIS': True, 'MAE': True, 'WAPE': True, 'sMAPE': True,
        'WIS_log_alta (pico)': True, 'WIS_log_entressafra': True, 'erro_semana_abs_medio': True}
linhas = []
for nv, g in UNICA.groupby('nivel'):
    g = g[~g.modelo.isin(['climatologia', 'Climatologia_sem'])]
    for c, asc in crit.items():
        if c not in g:
            continue
        o = g.dropna(subset=[c]).sort_values(c, ascending=asc).head(5)
        linhas.append(dict(nivel=nv, metrica=c, **{f'{i + 1}º': f'{m} ({v:.3g})' for i, (m, v) in enumerate(zip(o.modelo, o[c]))}))
    if 'erro_altura_mediano_pct' in g:
        o = g.assign(_k=g['erro_altura_mediano_pct'].abs()).dropna(subset=['_k']).sort_values('_k').head(5)
        linhas.append(dict(nivel=nv, metrica='|erro na altura do pico|', **{f'{i + 1}º': f'{m} ({v:.3g}%)' for i, (m, v) in enumerate(zip(o.modelo, o['erro_altura_mediano_pct']))}))
TOP5 = pd.DataFrame(linhas)
R2.salvar_tabela(TOP5, PASTA_AV2, 'top5_por_metrica')
TOP5
